# Tracking code before trackmate(Fiji)

#### Note: This code performs preliminary automated cell tracking and is not intended to produce fully accurate tracking results on its own. The tracking results may contain errors and require manual inspection and correction using TrackMate in Fiji.

## 1. Import libraries and data

In [ ]:
# Standard library
import os
import json
import random
from collections import defaultdict

# Numerical analysis and visualization
import numpy as np
import matplotlib.pyplot as plt

# Image processing
import tifffile as tiff
from PIL import Image, ImageDraw
from skimage import measure
from skimage.measure import label, regionprops

# Scientific computing and optimization
from scipy.spatial import distance
from scipy.optimize import linear_sum_assignment


In [ ]:
# Path of TIFF file
tiff_path = '../2_label_img.tif'

# import the TIFF file and read the image data
with tiff.TiffFile(tiff_path) as tif:
    data = tif.asarray()

# Print the shape of the data
print(f"Original shape: {data.shape}")

# make a list of mask images for each frame
masks = [data[t] for t in range(data.shape[0])]

## 2. Functions for tracking

In [ ]:
# make a list of labeled masks for each frame
def generate_labeled_masks(frame_masks):
    lbl_masks = []
    for t in range(frame_masks.shape[0]):
        lbl_mask = label(frame_masks[t])
        lbl_masks.append(lbl_mask)
    return lbl_masks

In [ ]:
# First, assign the nearest curr_label to each prev_label
# (only if the distance is within the threshold).
# Then, reorganize the assignments by curr_label.
#
# Directly assigning the nearest prev_label to each curr_label
# would prevent multiple prev_labels from being assigned to the same curr_label.
# Furthermore, assigning all prev_labels within the distance threshold
# could include prev_labels for which the curr_label is not the nearest match.

def get_assignments(prev_centroids, centroids, dist_threshold, new_lost_cue, track_history):
    assignments = []
    used_prev_labels = set()
    used_curr_labels = set()
    
    # Find the index-th nearest curr_label for each prev_label
    for prev_label, prev_centroid, index in prev_centroids:
        if prev_label in new_lost_cue:
            continue  # Skip prev_labels already in new_lost_cue
                
        distances = []
        for curr_label, curr_centroid in centroids:
            centroid_dist = distance.euclidean(prev_centroid, curr_centroid)
            if centroid_dist < dist_threshold:
                distances.append((centroid_dist, prev_label, prev_centroid, curr_label, curr_centroid, index))

        if distances:
            distances.sort()
            # Select the index-th nearest curr_label
            if index < len(distances):
                closest_curr = distances[index]
                assignments.append(closest_curr)
                used_curr_labels.add(closest_curr[3])
                used_prev_labels.add(prev_label)
            elif index == len(distances):
                if prev_label not in new_lost_cue:
                    new_lost_cue.append((prev_label, prev_centroid))
        
        else:
            if prev_label not in new_lost_cue:
                new_lost_cue.append((prev_label, prev_centroid))

    # Group prev_labels by curr_label
    reverse_assignments = defaultdict(list)
    for centroid_dist, prev_label, prev_centroid, curr_label, curr_centroid, index in assignments:
        reverse_assignments[curr_label].append((centroid_dist, prev_label, prev_centroid, curr_centroid, index))

    # Sort the prev_labels for each curr_label by centroid distance
    for curr_label in reverse_assignments:
        reverse_assignments[curr_label].sort(key=lambda x: x[0])

    # Add curr_labels with no matching prev_labels
    for curr_label, curr_centroid in centroids:
        if curr_label not in used_curr_labels:
            reverse_assignments[curr_label] = []

    # Convert the dictionary to a list
    reverse_assignments_list = list(reverse_assignments.items())

    return reverse_assignments_list, new_lost_cue, track_history


In [ ]:
# Check whether two cells can be connected based on their size and circularity
def check_cell_connect(track_history, curr_label, prev_label, props, connect_cell_size, connect_cell_circularity):
    prev_area = track_history[prev_label]['area']
    prev_perimeter = track_history[prev_label]['perimeter']
    
    curr_area = props[curr_label - 1].area 
    curr_perimeter = props[curr_label - 1].perimeter
    
    prev_circularity = (4 * np.pi * prev_area) / (prev_perimeter**2)
    curr_circularity = (4 * np.pi * curr_area) / (curr_perimeter**2)
    
    if(connect_cell_size[0] * prev_area <= curr_area <= connect_cell_size[1] * prev_area):
        if(connect_cell_circularity[0] * prev_circularity <= curr_circularity <= connect_cell_circularity[1] * prev_circularity):
            return True
    return False

In [ ]:
# Check cell division based on the combined area of two daughter cells
def check_cell_division(track_history, curr_label, best_label, second_best_label, props, size_ratio, new_check_division_list, is_chase_divided_cell):
    first_area = track_history[best_label]['area']
    second_area = track_history[second_best_label]["area"]
    parent_area = props[curr_label - 1].area 
    
    if is_chase_divided_cell:
        if(size_ratio[0] * parent_area <= first_area + second_area <= size_ratio[1] * parent_area):
            check_division_count = 0
            first_cell = (best_label, track_history[best_label]["centroid"], check_division_count)
            second_cell = (second_best_label, track_history[second_best_label]["centroid"], check_division_count)
            new_check_division_list.append(first_cell)
            new_check_division_list.append(second_cell)
            
            return True, new_check_division_list
        return False, new_check_division_list

    if(size_ratio[0] * parent_area <= first_area + second_area <= size_ratio[1] * parent_area):
        return True
    return False

In [ ]:
def handle_cell_division(track_history, lineage, curr_label, best_label, second_best_label, curr_centroid, props, next_label, t, new_labeled_mask, labeled_mask, new_centroids, used_prev, used_curr):
    # Mark best_label and second_best_label as used
    used_prev.add(best_label)
    used_prev.add(second_best_label)
    used_curr.add(curr_label)

    # Assign a new label to curr_label and add its centroid
    new_labeled_mask[labeled_mask == curr_label] = next_label
    index = 0
    new_centroids.append((next_label, curr_centroid, index))
    track_history[next_label] = {
        'centroid': curr_centroid,
        'missed': 0,
        'history': [(t, curr_centroid)],
        'area': props[curr_label - 1].area,
        'division': True,
        'daughters': [best_label, second_best_label],
        'parent': None,
        "perimeter": props[curr_label - 1].perimeter
    }
    
    track_history[best_label]['parent'] = next_label
    track_history[second_best_label]['parent'] = next_label
    
    # Add curr_label to the lineage with its new label
    lineage.append({
        'label': next_label,
        'frames': [t],
        'division': True,
        'daughters': [best_label, second_best_label],
        'parent': None
    })

    for cell in lineage:
        if cell['label'] == best_label:
            cell['parent'] = next_label
        if cell['label'] == second_best_label:
            cell['parent'] = next_label
    
    next_label += 1

    return next_label, track_history, lineage, new_centroids

In [ ]:
def update_check_division_list(check_division_list, new_check_division_list, unused_curr, track_history, lineage, check_div_dist_threshold, new_centroids, new_labeled_mask, labeled_mask, t):
    updated_check_division_list = []
    new_unused_curr = []
    check_division_list.extend(new_check_division_list)
    min_dist = check_div_dist_threshold
    used_unused_curr = set()
    distances = [] 
    assignments = [] 
    
    def update_division_count(div_label, div_centroid, check_division_count, updated_check_division_list):
        updated_check_division_list.append((div_label, div_centroid, check_division_count))
    
    for div_label, div_centroid, check_division_count in check_division_list:
        if check_division_count < 1:
            check_division_count +=1
            update_division_count(div_label, div_centroid, check_division_count, updated_check_division_list)
            continue 
        
        if check_division_count >= 3:
            continue
        
        for unused_label, unused_centroid in unused_curr:
            centroid_dist = distance.euclidean(div_centroid, unused_centroid)
            if centroid_dist < min_dist:
                matched_label = unused_label
                matched_centroid = unused_centroid
                distances.append((centroid_dist, matched_label, matched_centroid, div_label, div_centroid))
        # Select the nearest cell among the matched candidates
        if distances: # If one daughter cell is closer to both daughter cells in the previous frame, the other daughter cell may not be correctly connected. This requires manual correction.
            distances.sort()
            closest_matched = distances[0]
            _, matched_label, _, _, _ = closest_matched
            if matched_label not in used_unused_curr:
                used_unused_curr.add(matched_label)
            assignments.append(closest_matched)
            
        check_division_count +=1
        update_division_count(div_label, div_centroid, check_division_count, updated_check_division_list)
       
    # Prevent incorrect connections by selecting only the closest div_label for each matched_label
    if len (assignments) > 0:   
        reverse_assignments = defaultdict(list)
        for centroid_dist, matched_label, matched_centroid, div_label, div_centroid in assignments:
            reverse_assignments[matched_label].append((centroid_dist, matched_centroid, div_label, div_centroid))
        for matched_label in reverse_assignments:
            reverse_assignments[matched_label] = [min(reverse_assignments[matched_label], key=lambda x: x[0])]
        update_list = list(reverse_assignments.items())  # Convert the dictionary to a list
        
        # Collect div_labels included in update_list
        div_labels_to_remove = set()
        for matched_label, distances in update_list:
            if distances:  # Ensure the list is not empty
                _, _, div_label, _ = distances[0]
                div_labels_to_remove.add(div_label)

        # Remove entries with the corresponding div_labels from updated_check_division_list
        updated_check_division_list = [
            (div_label, div_centroid, check_division_count)
            for div_label, div_centroid, check_division_count in updated_check_division_list
            if div_label not in div_labels_to_remove
        ]
    
        # Update the tracking information using the selected connections
        for matched_label, distances in update_list:
            centroid_dist, matched_centroid, div_label, div_centroid = distances[0]
                
            # Check if div_label is in track_history and has a "parent" key
            if div_label in track_history:
                uncorrect_parent_id = track_history[div_label]["parent"]
                
                if uncorrect_parent_id is not None:
                    track_history[uncorrect_parent_id]["daughters"] = []
                    track_history[uncorrect_parent_id]["division"] = False

                    for cell in lineage:
                        if cell["label"] == uncorrect_parent_id:
                            cell["daughters"] = []
                            cell["division"] = False
                else:
                    print(f"parent of div_label '{div_label}' is already modified.")
            else:
                # print(f"Warning: div_label '{div_label}' not found in track_history")
                print(f"\033[91mWarning: div_label '{div_label}' not found in track_history\033[0m")
        
            index = 0
            new_labeled_mask[labeled_mask == matched_label] = div_label
            new_centroids.append((div_label, matched_centroid, index))
            track_history[div_label]['centroid'] = matched_centroid
            track_history[div_label]['missed'] = 0
            track_history[div_label]['history'].append((t, matched_centroid))
            track_history[div_label]['parent'] = None
            for cell in lineage:
                if cell["label"] == div_label:
                    cell["frames"].append(t)
                    cell["parent"] = None     
    
    # Disconnect any remaining incorrect parent connections involving only one daughter cell
    for div_label, div_centroid, _ in updated_check_division_list:
        verify_parent_id = track_history.get(div_label, {}).get("parent")
        if verify_parent_id is not None:
            verify_daughters = track_history.get(verify_parent_id, {}).get("daughters", [])
            
            if not verify_daughters:  # If the list is empty or None
                track_history[div_label]['parent'] = None
                for cell in lineage:
                    if cell["label"] == div_label:
                        cell["parent"] = None
        else:
            # Handle cases where no parent is assigned to div_label
            print(f"div_label {div_label}'s parent key is already set to None.")
        
    # Remove unused_curr cells that have already been assigned
    for label, centroid in unused_curr:
        if label not in used_unused_curr:
            new_unused_curr.append((label, centroid))
    unused_curr = new_unused_curr 
    
    return updated_check_division_list, track_history, lineage, new_centroids, new_labeled_mask, unused_curr

In [ ]:
def process_frame(track_history, lineage, prev_centroids, centroids, props, dist_threshold, t, new_labeled_mask, labeled_mask, new_centroids, next_label, size_ratio, max_gap, lost_cue, connect_cell_size, gap_dist_threshold, check_division_list, check_div_dist_threshold, connect_cell_circularity):
    used_prev = set()
    used_curr = set()
    unused_curr = []
    new_lost_cue = []
    new_check_division_list = []

    def update_track_history_and_lineage(curr_label, prev_label, curr_centroid, props):
        index = 0
        new_labeled_mask[labeled_mask == curr_label] = prev_label
        new_centroids.append((prev_label, curr_centroid, index))
        track_history[prev_label]['centroid'] = curr_centroid
        track_history[prev_label]['missed'] = 0
        track_history[prev_label]['history'].append((t, curr_centroid))  # Update last entry
        track_history[prev_label]["area"] = props[curr_label - 1].area
        track_history[prev_label]["perimeter"] = props[curr_label - 1].perimeter
        for cell in lineage:
            if cell['label'] == prev_label:
                cell['frames'].append(t)  # Update last entry

    def update_centroids(prev_label, prev_centroid, index, updated_prev_centroids):
        updated_prev_centroids.append((prev_label, prev_centroid, index))
        
    def update_missed_count(prev_label, track_history):
        track_history[prev_label]['missed'] += 1

    def process_assignments(assignments, max_iterations, new_lost_cue, unused_curr, track_history, centroids, lineage, new_centroids, props):
        nonlocal next_label
        nonlocal new_check_division_list
        iteration = 0

        # Determine the optimal cell assignments between consecutive frames
        while True:
            if iteration >= max_iterations:
                print("iteration reached max_iteration!")
                break
            
            old_assignments = assignments.copy()
            updated_prev_centroids = []
            for curr_label, distances in assignments: 
                if len(distances) == 1:
                    _, prev_label, prev_centroid, _, index = distances[0]
                    
                    connect = check_cell_connect(track_history, curr_label, prev_label, props, connect_cell_size, connect_cell_circularity)
                    
                    if connect == True:
                        update_centroids(prev_label, prev_centroid, index, updated_prev_centroids)
                    elif connect == False:
                        index+1
                        update_centroids(prev_label, prev_centroid, index, updated_prev_centroids)
                    
                elif len(distances) >= 2:
                    best_match = distances[0]
                    second_best_match = distances[1]
                    _, best_label, best_centroid, _, best_index = best_match
                    _, second_best_label, second_best_centroid, _, second_best_index = second_best_match
            
                   
                    division = check_cell_division(track_history, curr_label, best_label, second_best_label, props, size_ratio, new_check_division_list, is_chase_divided_cell=False)

                    if division == True:
                        update_centroids(best_label, best_centroid, best_index, updated_prev_centroids)
                        update_centroids(second_best_label, second_best_centroid, second_best_index, updated_prev_centroids)
                    
                    elif division == False:
                        update_centroids(best_label, best_centroid, best_index, updated_prev_centroids)
                        second_best_index += 1
                        update_centroids(second_best_label, second_best_centroid, second_best_index, updated_prev_centroids)
                            
                    for _, prev_label, prev_centroid, _, index in distances[2:]:
                        index += 1
                        update_centroids(prev_label, prev_centroid, index, updated_prev_centroids)
            
            new_assignments, new_lost_cue, track_history = get_assignments(updated_prev_centroids, centroids, dist_threshold, new_lost_cue, track_history)
            
            if new_assignments == old_assignments:
                break
            
            assignments = new_assignments
            
            iteration += 1

        # Connect cells based on the finalized assignments
        for curr_label, distances in assignments:
            if len(distances) == 1:
                _, prev_label, prev_centroid, curr_centroid, _ = distances[0]
                
                connect = check_cell_connect(track_history, curr_label, prev_label, props, connect_cell_size, connect_cell_circularity)
                
                if connect == True:
                    if prev_label not in used_prev:
                        used_curr.add(curr_label)
                        used_prev.add(prev_label)
                        update_track_history_and_lineage(curr_label, prev_label, curr_centroid, props)
                elif connect == False:
                    if prev_label not in used_prev:
                        new_lost_cue.append((prev_label, prev_centroid)) 
                    curr_centroid = [centroid for label, centroid in centroids if label == curr_label]
                    unused_curr.append((curr_label, curr_centroid[0]))                                          
                    
            elif len(distances) >= 2:
                best_match = distances[0]
                second_best_match = distances[1]
                _, best_label, _, curr_centroid,_ = best_match
                _, second_best_label, second_best_centroid, curr_centroid, _ = second_best_match
                
                division, new_check_division_list = check_cell_division(track_history, curr_label, best_label, second_best_label, props, size_ratio, new_check_division_list, is_chase_divided_cell=True)

                if division == True:
                    next_label, track_history, lineage, new_centroids = handle_cell_division(track_history, lineage, curr_label, best_label, second_best_label, curr_centroid, props, next_label, t, new_labeled_mask, labeled_mask, new_centroids, used_prev, used_curr)
                elif division == False:
                    if best_label not in used_prev:
                        used_curr.add(curr_label)
                        used_prev.add(best_label)
                        update_track_history_and_lineage(curr_label, best_label, curr_centroid, props)

                    if second_best_label not in used_prev:
                        new_lost_cue.append((second_best_label, second_best_centroid))
                        
                for _, prev_label, prev_centroid, curr_centroid, _ in distances[2:]:
                    if prev_label not in used_prev:
                        new_lost_cue.append((prev_label, prev_centroid))
                        
            elif len(distances) == 0:
                curr_centroid = [centroid for label, centroid in centroids if label == curr_label]
                unused_curr.append((curr_label, curr_centroid[0]))

    # Obtain and process the initial assignments
    assignments, new_lost_cue, track_history = get_assignments(prev_centroids, centroids, dist_threshold, new_lost_cue, track_history)
    process_assignments(assignments, 10, new_lost_cue, unused_curr, track_history, centroids, lineage, new_centroids, props)

    # Check whether cells in check_division_list can be matched with unused cells.
    # If a match is found, cancel the division assignment and reconnect the cells.
    updated_check_division_list, track_history, lineage, new_centroids, new_labeled_mask, unused_curr = update_check_division_list(check_division_list, new_check_division_list, unused_curr, track_history, lineage, check_div_dist_threshold, new_centroids, new_labeled_mask, labeled_mask, t)
    
    # Gap closing
    lost_cue.extend(new_lost_cue)
    updated_lost_cue = []
    used_unused_curr = set()

    for prev_label, prev_centroid in lost_cue:
        matched_centroids = []
        matched_label = None
        min_dist = gap_dist_threshold
        
        if track_history[prev_label] == False:
            print("something wrong!")
            continue
        
        if track_history[prev_label]["missed"] > 0 and track_history[prev_label]["missed"] <= max_gap:
            for curr_label, curr_centroid in unused_curr:
                centroid_dist = distance.euclidean(prev_centroid, curr_centroid)
                if centroid_dist < min_dist:
                    matched_label = curr_label
                    matched_centroid = curr_centroid
                    
                    connect = check_cell_connect(track_history, matched_label, prev_label, props, connect_cell_size, connect_cell_circularity)
                    if connect == True:
                        matched_centroids.append((centroid_dist, matched_label, matched_centroid))

            if matched_centroids:
                matched_centroids.sort()
                _, closest_matched_label, closest_matched_centroid = matched_centroids[0] # Select the nearest candidate for gap closing to avoid assigning cells solely based on processing order. Further improvement is needed to resolve competing matches.
                used_unused_curr.add(closest_matched_label)
                update_track_history_and_lineage(closest_matched_label, prev_label, closest_matched_centroid, props)
            
            else:
                update_missed_count(prev_label, track_history)
                
                if track_history[prev_label]["missed"] <= max_gap:
                    updated_lost_cue.append((prev_label, prev_centroid))
                       
        else: 
            update_missed_count(prev_label, track_history)
            
            if track_history[prev_label]["missed"] <= max_gap:
                updated_lost_cue.append((prev_label, prev_centroid))
    
    for curr_label, curr_centroid in unused_curr:
        if curr_label not in used_unused_curr:
            index = 0
            new_labeled_mask[labeled_mask == curr_label] = next_label
            new_centroids.append((next_label, curr_centroid, index))
            
            track_history[next_label] = {
                'centroid': curr_centroid,
                'missed': 0,
                'history': [(t, curr_centroid)],
                'area': props[curr_label - 1].area,
                'division': False,
                'daughters': [],
                'parent': None,
                "perimeter": props[curr_label - 1].perimeter
                }
            
            lineage.append({
                'label': next_label,
                'frames': [t],
                'division': False,
                'daughters': [],
                'parent': None
                })
            
            next_label += 1

    return track_history, lineage, next_label, updated_lost_cue, used_prev, used_curr, new_labeled_mask, new_centroids, updated_check_division_list


In [ ]:
# Track cells backward from the last frame to the first frame
def track_cells_backward(frames, labeled_masks, dist_threshold=25, size_ratio=(0.75, 1.33), max_gap=3, connect_cell_size=(0.57, 1.75), gap_dist_threshold=35, check_div_dist_threshold=25, connect_cell_circularity=(0.75, 1.33)):
    track_history = {}
    lineage = []
    next_label = 1
    last_centroids = []
    tracked_centroids = []
    lost_cue = []
    check_division_list = []

    # Initialize track_history and lineage for the last frame
    last_frame = len(frames) - 1
    last_mask = labeled_masks[last_frame]
    last_props = regionprops(last_mask)
    for prop in last_props:
        track_history[next_label] = {
            'centroid': prop.centroid,
            'missed': 0,
            'history': [(last_frame, prop.centroid)],
            'area': prop.area,
            'division': False,
            'daughters': [],
            'parent': None,
            "perimeter": prop.perimeter
        }
        lineage.append({
            'label': next_label,
            'frames': [last_frame],
            'division': False,
            'daughters': [],
            'parent': None
        })
        index = 0
        last_centroids.append((next_label, prop.centroid, index))
        next_label += 1
    tracked_centroids.append(last_centroids)
    print("last frame was processed.")

    # Track cells backward through frames
    for t in range(last_frame - 1, -1, -1):
        # prev_centroids refers to cells in the temporally later frame.
        # curr_centroids refers to cells in the temporally earlier frame, where the cells in prev_centroids are expected to be matched.
        prev_centroids = tracked_centroids[last_frame - 1 - t]
        curr_mask = labeled_masks[t]
        curr_props = regionprops(curr_mask)
        curr_centroids = [(prop.label, prop.centroid) for prop in curr_props] 

        new_labeled_mask = np.zeros_like(curr_mask)
        new_centroids = []
        
        updated_lost_cue = []

        track_history, lineage, next_label, updated_lost_cue, _, _, new_labeled_mask, new_centroids, updated_check_division_list = process_frame(
            track_history, lineage, prev_centroids, curr_centroids, curr_props, dist_threshold, t, new_labeled_mask, curr_mask, new_centroids, next_label, size_ratio, max_gap, lost_cue, connect_cell_size, gap_dist_threshold, check_division_list, check_div_dist_threshold, connect_cell_circularity)

        # Handle lost cues
        if len(updated_lost_cue) >= 1:
            lost_cue = updated_lost_cue
            # print("content(s) of lost_cue:", lost_cue)  
            print("content(s) of lost_cue:",
                  [(label, tuple(round(float(c), 2) for c in centroid))
                    for label, centroid in lost_cue]) 
            
        # Handle check_division_list 
        if len(updated_check_division_list) >= 1:
            check_division_list = updated_check_division_list
            # print("content(s) of check_division_list:", check_division_list) 
            print("content(s) of check_division_list:",
                  [(label, tuple(round(float(c), 2) for c in centroid), count)
                   for label, centroid, count in check_division_list])
        
        tracked_centroids.append(new_centroids)
        print(f"frame_{t} was processed.")

        # Update the labeled masks for the current frame
        labeled_masks[t] = new_labeled_mask

    return track_history, lineage, labeled_masks


## 3. Run tracking

In [ ]:
# Generate labeled masks
labeled_masks = generate_labeled_masks(data)

# Perform backward cell tracking
track_history, lineages, labeled_masks = track_cells_backward(masks, labeled_masks)

## 4. Visualization of Cell Tracking Results

In [ ]:
# Generate a random RGB color
def generate_random_color():
    return tuple(random.choices(range(256), k=3))

# Create a dictionary of colors for each cell label
colors = {label: generate_random_color() for label in track_history.keys()}

# Set up the output directory for high-resolution tracking visualizations
output_dir = 'tracked_images'
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, 'all_tracked_frames.tif')

# Visualize and save the tracking results for each frame in TIF format
for t, mask in enumerate(masks):
    fig, ax = plt.subplots(figsize=(10, 10), dpi=300)
    ax.imshow(mask, cmap='gray')
    
    for label, cell_history in track_history.items():
        if len(cell_history['history']) > 0:

            for i in range(len(cell_history['history'])):
                t1, (y1, x1) = cell_history['history'][i]
                
                # Draw a line connecting daughter and parent cells
                if i == len(cell_history['history']) - 1  and cell_history['parent'] is not None and t1 == t:
                    parent_label = cell_history['parent']
                    parent_history = track_history[parent_label]['history']
                    parent_t, (parent_y, parent_x) = parent_history[0]
                    color = np.array(colors[label]) / 255
                    ax.plot([x1, parent_x], [y1, parent_y], color=color, linestyle='solid', linewidth=1)
                
                # Draw lines connecting cell positions across consecutive frames
                if i < len(cell_history['history']) - 1:  # Access the next element only if it exists
                    t2, (y2, x2) = cell_history['history'][i+1]
                    # Draw the trajectory if t1 is later than t2 and matches the current frame
                    if t1 > t2 and t1 == t:
                        color = np.array(colors[label]) / 255  # Convert RGB values to the appropriate range
                        ax.plot([x1, x2], [y1, y2], color=color, linewidth=1)
                        ax.plot(x1, y1, 'o', color=color, markersize=2)
                        
                 # Draw the first cell position (the last element in the history list)
                elif i == len(cell_history['history']) - 1 and t1 == t:
                    color = np.array(colors[label]) / 255
                    ax.plot(x1, y1, 'o', color=color, markersize=2)

    plt.title(f'Tracked Frame {t}')
    plt.axis('off')
    plt.savefig(os.path.join(output_dir, f'tracked_frame_{t:03d}.tif'), dpi=300, bbox_inches='tight')
    plt.close()

## 5. Preparation of TRKS Data

In [ ]:
# Prepare data for TRKS export
trks_data = {
    'lineages': lineages,
    'y': np.expand_dims(np.array(labeled_masks), axis=3).tolist()  # (t, x, y, 1)
}

In [ ]:
# Save the TRKS file
with open('output.trks', 'w') as f:
    json.dump(trks_data, f)

print("TRKS file created successfully.")

## 6. Generation of Lineage-Based Color Maps

In [ ]:
# Generate a color map based on cell lineage information
def get_color_map(lineages, colormap='grayscale'):
    # Generate a color map that assigns the same color to cells belonging to the same lineage.
    color_map = {}
    processed = set()

    def assign_color(cell):
        label = cell['label']
        parent_id = cell['parent']

        if label in processed:
            return
        
        # Assign a color to the parent cell first, if one exists
        if parent_id is not None:
            parent_cell = next((c for c in lineages if c['label'] == parent_id), None)
            if parent_cell:
                assign_color(parent_cell)
                color_map[label] = color_map[parent_id]
            else:
                # Handle cases where the parent cell is not found
                print(f"Warning: Parent cell with id {parent_id} not found in lineages")
                return
        else:
            # Assign a new color to cells without a parent
            if colormap == 'grayscale':
                base_color = np.random.randint(2**8, 2**16 - 1)  # Random grayscale intensity
            else:
                base_color = np.random.randint(0, 256, 3)  # Random RGB color
            color_map[label] = base_color
        
        processed.add(label)

    for cell in lineages:
        assign_color(cell)

    return color_map

In [ ]:
# Save lineage-colored label images
def save_label_images(y, lineages, output_dir, colormap='grayscale'):
    # Save labeled images for all frames, assigning different colors to cell lineages.
    # Cells belonging to the same lineage are assigned the same color.
    color_map = get_color_map(lineages, colormap)

    for frame_index in range(y.shape[0]):
        frame_y = y[frame_index, :, :, 0]  # Assume a single-channel image
        if colormap == 'grayscale':
            frame_y_color = np.zeros(frame_y.shape, dtype=np.uint16)
        else:
            frame_y_color = np.zeros((*frame_y.shape, 3), dtype=np.uint8)

        unique_labels = np.unique(frame_y)

        for label in unique_labels:
            if label == 0:  # Background
                continue

            if label in color_map:
                color = color_map[label]
            else:
                print("something went wrong!!")
                print(label)
                color = np.random.randint(2**8, 2**16 - 1) if colormap == 'grayscale' else np.random.randint(0, 256, 3)
                color_map[label] = color

            frame_y_color[frame_y == label] = color

        # Save the image as a TIF file
        output_path = os.path.join(output_dir, f'label_frame_{frame_index}.tif')
        img_pil = Image.fromarray(frame_y_color)
        img_pil.save(output_path)

In [ ]:
# Load the predicted TRKS file
# with open('output.trks', 'r') as f:
#     output_res_data = json.load(f)

y_test = np.array(trks_data['y'])
lineages_test = trks_data['lineages']

## 7. Export of Lineage-Colored Label Images

In [ ]:
# Specify the output directory
output_dir = './prediction'
os.makedirs(output_dir, exist_ok=True)

# Save all frames as grayscale images
save_label_images(y_test, lineages_test, output_dir, colormap='grayscale')

# Save all frames as RGB images
output_dir_rgb = './prediction_rgb'
os.makedirs(output_dir_rgb, exist_ok=True)
save_label_images(y_test, lineages_test, output_dir_rgb, colormap='rgb')

print("Label images saved successfully.")